<a href="https://colab.research.google.com/github/justorfc/Estadistica_Aplicada_con_Python_y_R/blob/main/10_Semana_10_Regresi%C3%B3n_Lineal_M%C3%BAltiple_y_Supuestos_del_Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Aquí tienes la propuesta estructurada para la **Semana 10**. En esta etapa del **Eje IV**, avanzamos de la regresión simple a la **Regresión Lineal Múltiple**, un escenario mucho más realista en la ingeniería agrícola donde múltiples factores interactúan simultáneamente para determinar un resultado. Además, introducimos la auditoría crítica del modelo: el análisis de los residuos.

# Semana 10: Regresión Lineal Múltiple y Supuestos del Modelo

**Resultado de aprendizaje:** Construye e interpreta modelos de regresión múltiple, evaluando el efecto simultáneo de variables agroclimáticas, y diagnostica la validez del modelo mediante el análisis de residuos y multicolinealidad.

---

#### Sesión 1: El efecto simultáneo y la Multicolinealidad (80 - 90 minutos)

**Objetivo:** Comprender cómo aislar el efecto de una variable mientras las demás se mantienen constantes (*ceteris paribus*) y ajustar un modelo múltiple en Python.

* **20 min - Diálogo socrático y análisis de interacciones (Lápiz y papel):**
* *Situación:* El rendimiento no solo depende del agua. Si regamos mucho pero el suelo es pobre en materia orgánica, la planta no rinde igual. Si metemos agua y materia orgánica en la misma ecuación matemática, ¿cómo sabemos quién aporta más?
* *Actividad:* Discusión sobre el riesgo de la **Multicolinealidad**. ¿Qué pasa si intentamos predecir la evapotranspiración metiendo en el modelo la "Temperatura en °C" y la "Temperatura en °F"? El modelo colapsa por redundancia.


* **45 min - Exploración en Google Colab (Python):**
* Carga del cuaderno de la semana 10.
* Ajuste de un modelo múltiple usando `statsmodels` con la fórmula `Y ~ X1 + X2 + X3`.
* Interpretación del **$R^2$ Ajustado**, el cual penaliza al modelo si le metemos variables basura que no aportan información real.


* **15 min - Reflexión manuscrita:**
* Interpretación técnica de un coeficiente múltiple: "Por cada 1% extra de materia orgánica, el rendimiento aumenta en $\beta$ toneladas, *asumiendo que el riego y la temperatura no cambien*".



---

#### Sesión 2: Análisis de Residuos y Transición a R (80 - 90 minutos)

**Objetivo:** Aprender que un modelo solo es válido si sus errores (residuos) son aleatorios, y trasladar esta auditoría paramétrica a R.

* **20 min - Los Supuestos de la Regresión:**
* Explicación en pizarra de los tres pilares: Normalidad de los residuos, Homocedasticidad (varianza constante del error) e Independencia.


* **25 min - Prompts para modelos múltiples y diagnósticos en R:**
* Demostración de cómo instruir al asistente de IA para ajustar una regresión múltiple con `lm(Y ~ X1 + X2 + X3)` en R.
* La magia de la función `plot(modelo)` en R base, que genera automáticamente 4 gráficos diagnósticos de residuos espectaculares sin esfuerzo.


* **40 min - Reto en Posit Cloud:**
* Los estudiantes replican el modelo múltiple en RMarkdown, extraen el $R^2$ ajustado, visualizan los gráficos de residuos y documentan sus conclusiones en la bitácora de IA.



---

A continuación, el contenido listo para integrarse en las celdas de tu cuaderno de Google Colab.

---

### Celda de Texto 1

```markdown
# Semana 10: Regresión Lineal Múltiple y Supuestos
**Asignatura:** Estadística Aplicada con Python y R  
**Programa:** Ingeniería Agrícola - Universidad de Sucre  
**Profesor:** Justo Rafael Fuentes Cuello  

---

### Situación de Interés: La Realidad Multivariable
La semana pasada predijimos el rendimiento del maíz usando únicamente la lámina de riego. Sin embargo, en el campo real, el rendimiento es el resultado de una "orquesta" de variables.

Hoy vamos a construir un **Modelo de Regresión Múltiple** para predecir el Rendimiento (ton/ha) combinando tres factores simultáneos: **Lámina de Riego (mm)**, **Materia Orgánica (%)** y **Temperatura Promedio (°C)**. Aprenderemos a interpretar qué variable tiene mayor peso y a auditar si las matemáticas de nuestro modelo son confiables analizando sus "residuos" (errores).

```

### Celda de Código 1

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid")
np.random.seed(42)

# Simulamos datos de 60 parcelas con tres variables independientes
riego_mm = np.random.uniform(300, 700, 60)
materia_organica_pct = np.random.uniform(1.0, 5.0, 60)
temperatura_c = np.random.uniform(24, 32, 60)

# El rendimiento verdadero depende de las tres variables + un error aleatorio
# Fórmula real oculta: Base + Efecto Riego + Efecto MO - Efecto Calor Excesivo
error = np.random.normal(loc=0, scale=0.6, size=60)
rendimiento = 1.0 + (0.008 * riego_mm) + (0.9 * materia_organica_pct) - (0.15 * temperatura_c) + error

df_multivariable = pd.DataFrame({
    'Riego_mm': riego_mm,
    'Materia_Organica': materia_organica_pct,
    'Temperatura': temperatura_c,
    'Rendimiento': rendimiento
})

print("Dataset multivariable cargado.")
df_multivariable.head()

### 1. El Riesgo de la Multicolinealidad
Antes de meter todas las variables a la licuadora matemática, debemos revisar que no estén altamente correlacionadas *entre sí*.
Si incluimos dos variables que explican exactamente lo mismo (ej. precipitación anual y precipitación semestral de la misma zona), el modelo sufrirá de **Multicolinealidad** y sus predicciones colapsarán.

Verificamos esto con una matriz de correlación. Queremos que las variables predictoras (X) estén correlacionadas con el Rendimiento (Y), pero NO muy correlacionadas entre sí.

```

### Celda de Código 2

In [ ]:
# Matriz de correlación
correlaciones = df_multivariable.corr()

plt.figure(figsize=(6, 4))
sns.heatmap(correlaciones, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f")
plt.title('Matriz de Correlación: Buscando Multicolinealidad')
plt.show()

print("Observación: Riego, Materia_Organica y Temperatura tienen correlaciones muy bajas entre sí (cercanas a 0).")
print("Esto es excelente: cada variable aporta información única e independiente al modelo.")

### 2. Ajuste del Modelo de Regresión Múltiple (OLS)
Ahora, usamos la fórmula estadística indicando que el Rendimiento depende de la suma de los tres factores: `Rendimiento ~ Riego_mm + Materia_Organica + Temperatura`.

```

### Celda de Código 3

In [ ]:
# Ajustamos el modelo múltiple
modelo_multiple = smf.ols(
    formula='Rendimiento ~ Riego_mm + Materia_Organica + Temperatura',
    data=df_multivariable
).fit()

# Mostramos el resumen
print(modelo_multiple.summary())

### 3. Diagnóstico del Modelo: El Análisis de Residuos
El "Residuo" es el error del modelo: la diferencia entre lo que el modelo predijo y el rendimiento real medido en campo.
Para que podamos confiar en un modelo, sus errores deben ser simple "ruido blanco" (aleatorios).

**Supuesto de Homocedasticidad:** La varianza de los residuos debe ser constante. Si graficamos las predicciones vs. los residuos, los puntos deben verse como una nube dispersa sin forma de "embudo" o "curva".

```

### Celda de Código 4

In [ ]:
# Extraemos las predicciones del modelo y sus respectivos errores (residuos)
predicciones = modelo_multiple.fittedvalues
residuos = modelo_multiple.resid

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Gráfico 1: Predicciones vs Residuos (Prueba de Homocedasticidad)
sns.scatterplot(x=predicciones, y=residuos, color='purple', ax=axes[0], s=60)
axes[0].axhline(0, color='black', linestyle='--')
axes[0].set_title('Predicciones vs. Residuos')
axes[0].set_xlabel('Rendimiento Predicho (ton/ha)')
axes[0].set_ylabel('Residuo (Error en ton/ha)')

# Gráfico 2: Histograma de los Residuos (Prueba de Normalidad)
sns.histplot(residuos, kde=True, color='purple', ax=axes[1])
axes[1].set_title('Distribución de los Errores (Normalidad)')
axes[1].set_xlabel('Residuo (ton/ha)')

plt.tight_layout()
plt.show()

### 🛑 Reflexión y Reserva Cognitiva (Síntesis manuscrita)
Toma lápiz y papel, analiza la tabla resumen de `statsmodels` y las gráficas, y responde:
1. Revisa el coeficiente de la variable `Materia_Organica`. ¿Qué valor tiene? Escribe una frase interpretando exactamente qué significa ese número para el agricultor en términos de toneladas por hectárea.
2. Compara el coeficiente de la `Temperatura`. ¿Por qué es un número negativo? ¿Qué sentido agronómico tiene este resultado matemático en relación al estrés térmico?
3. En la regresión múltiple no miramos el "$R^2$ normal", miramos el **$R^2$ Ajustado (Adj. R-squared)**. Investiga brevemente y escribe con tus propias palabras por qué el $R^2$ normal es un "mentiroso" cuando le agregamos demasiadas variables al modelo.

---

### Instrucciones para el reto en R (Trabajo Autónomo y Sesión 2)

**Misión:** Has logrado predecir un sistema multivariable. Tu reto es llevar esta regresión a **R** en **Posit Cloud**, donde el diagnóstico de residuos es nativamente superior y requiere una sola línea de código.

**Pasos a seguir:**
1. Abre tu proyecto en Posit Cloud y crea un nuevo documento RMarkdown.
2. Utiliza este *prompt* para guiar a tu agente de IA:
   > *"Actúa como un estadístico analizando datos en R. En Python ajustamos un modelo de regresión múltiple usando statsmodels con la fórmula 'Rendimiento ~ Riego + Materia_Organica + Temperatura'. Necesito replicar esto en R. Escribe el código para simular este dataframe con tres variables independientes. Luego, muéstrame cómo usar `lm()` para ajustar el modelo múltiple y `summary()` para extraer el R-cuadrado ajustado y los coeficientes. Finalmente, enséñame el 'truco' de usar la función base `plot(modelo)` en R para generar automáticamente los gráficos diagnósticos de residuos (Normal Q-Q y Residuals vs Fitted). Explícalo paso a paso."*
3. Al ejecutar `plot(modelo)` en R, verás que la consola te pide presionar "Enter" para pasar de un gráfico a otro. Pídele al chatbot que te enseñe cómo usar `par(mfrow = c(2, 2))` para que los 4 gráficos de residuos salgan juntos en una sola imagen dentro de tu RMarkdown.
4. **Entrega:** Renderiza tu documento. En tu "Bitácora de IA", reporta cómo R simplificó la auditoría del modelo frente al código manual que tuvimos que escribir en Python para graficar los residuos.